# 00 — Cache and Run Control

This notebook controls demo runtime mode:
- **Fast demo**: load existing artifacts only.
- **Full recomputation**: run the full script chain only if required artifacts are missing.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TABLES = ROOT / 'project report' / 'tables'
GRAPHICS = ROOT / 'project report' / 'graphics'
PROCESSED = ROOT / 'data' / 'processed'
DB_PATH = ROOT / 'bond_rates_database.db'

print(f'ROOT: {ROOT}')
print(f'DB exists: {DB_PATH.exists()}')

ROOT: C:\Users\sotov\Documents\Interdependence of Ruble and Yuan Borrowing rates
DB exists: True


In [6]:
tier0_required = [
    DB_PATH,
]

tier1_required = [
    TABLES / 'tab_5_1_descriptive.tex',
    TABLES / 'tab_5_2_adf.tex',
    TABLES / 'tab_5_3_coint.tex',
    TABLES / 'tab_5_4_granger.tex',
    TABLES / 'tab_5_5_var.tex',
    TABLES / 'tab_5_6_ols.tex',
    TABLES / 'tab_6_1_ns_metrics.tex',
    TABLES / 'tab_6_2_abnormal.tex',
    GRAPHICS / 'fig_4_2_russian_rates.png',
    GRAPHICS / 'fig_4_3_lpr_fx.png',
    GRAPHICS / 'fig_4_5_chinese_bonds.png',
    GRAPHICS / 'fig_4_7_global.png',
    GRAPHICS / 'fig_5_1_correlation.png',
    GRAPHICS / 'fig_5_2_irf.png',
    GRAPHICS / 'fig_6_1_spreads.png',
    GRAPHICS / 'fig_6_2_ns_residuals.png',
]

tier2_required = [
    PROCESSED / 'signals_latest.csv',
    PROCESSED / 'signals_latest_snapshot.csv',
    PROCESSED / 'signals_latest_diagnostics.csv',
    PROCESSED / 'portfolio_backtest_summary.csv',
    PROCESSED / 'ml_signals.csv',
    PROCESSED / 'ml_signals_snapshot.csv',
    PROCESSED / 'ml_signals_diagnostics.csv',
    PROCESSED / 'ml_backtest_summary.csv',
    TABLES / 'tab_6_3_model_comparison_ru.csv',
    TABLES / 'tab_6_3_model_comparison_cn.csv',
    TABLES / 'tab_6_3_model_comparison_best_model.csv',
    TABLES / 'tab_6_3_model_comparison_avg_rmse.png',
    TABLES / 'tab_6_4_advanced_methods.csv',
]

notebook_deps = {
    '01_data_and_statistics_demo.ipynb': tier1_required,
    '02_forecasting_and_spreads_demo.ipynb': [
        GRAPHICS / 'fig_6_1_spreads.png',
        GRAPHICS / 'fig_6_2_ns_residuals.png',
        TABLES / 'tab_6_1_ns_metrics.tex',
        TABLES / 'tab_6_2_abnormal.tex',
        TABLES / 'tab_6_3_model_comparison_ru.csv',
        TABLES / 'tab_6_3_model_comparison_cn.csv',
    ],
    '03_signals_demo_classical_vs_ml.ipynb': [
        PROCESSED / 'signals_latest_snapshot.csv',
        PROCESSED / 'signals_latest_diagnostics.csv',
        PROCESSED / 'portfolio_backtest_summary.csv',
        PROCESSED / 'ml_signals_snapshot.csv',
        PROCESSED / 'ml_signals_diagnostics.csv',
        PROCESSED / 'ml_backtest_summary.csv',
        TABLES / 'tab_6_4_advanced_methods.csv',
    ],
    '04_portfolio_and_risk_demo.ipynb': [
        PROCESSED / 'portfolio_backtest_summary.csv',
        PROCESSED / 'signals_latest_snapshot.csv',
        PROCESSED / 'ml_backtest_summary.csv',
    ],
    '99_full_pipeline_dashboard.ipynb': tier1_required + tier2_required,
}

def missing(paths):
    return [p for p in paths if not p.exists()]

m0 = missing(tier0_required)
m1 = missing(tier1_required)
m2 = missing(tier2_required)

print(f'Tier-0 missing: {len(m0)}')
print(f'Tier-1 missing: {len(m1)}')
print(f'Tier-2 missing: {len(m2)}')
if m2:
    for p in m2[:12]:
        print('  -', p)

fast_demo_ready = len(m2) == 0
print(f'Fast demo ready: {fast_demo_ready}')

notebook_status = []
for nb, deps in notebook_deps.items():
    miss = missing(deps)
    notebook_status.append({
        'notebook': nb,
        'ready': len(miss) == 0,
        'missing_count': len(miss),
        'missing_preview': ', '.join(p.name for p in miss[:3]) if miss else ''
    })

status_df = pd.DataFrame(notebook_status)
print('\nPer-notebook readiness:')
display(status_df)

Tier-0 missing: 0
Tier-1 missing: 0
Tier-2 missing: 0
Fast demo ready: True


NameError: name 'pd' is not defined

In [ ]:
def run_cmd(cmd):
    print('\n>>', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(ROOT), text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')

def run_full_chain():
    chain = [
        ['python', 'scripts/run_pipeline.py'],
        ['python', 'scripts/check_data_status.py'],
        ['python', 'scripts/run_model_comparison.py', '--curve', 'both', '--horizon', '1', '--train-window', '60', '--export-prefix', 'project report/tables/tab_6_3_model_comparison'],
        ['python', 'scripts/run_signals.py', '--save-csv', 'data/processed/signals_latest.csv'],
        ['python', 'scripts/run_ml_signals.py', '--save-prefix', 'data/processed/ml_signals'],
        ['python', 'scripts/run_portfolio_backtest.py'],
        ['python', 'scripts/run_ml_backtest.py'],
        ['python', 'scripts/run_advanced_comparison.py', '--output-prefix', 'project report/tables/tab_6_4_advanced_methods'],
        ['python', 'scripts/export_report_data.py', '--with-model-comparison', '--with-portfolio-backtest', '--with-advanced-comparison', '--with-ml-signals'],
    ]
    for cmd in chain:
        run_cmd(cmd)

# Set to True to force recomputation even when cache exists.
FORCE_FULL = False
if FORCE_FULL or not fast_demo_ready:
    print('Running full recomputation...')
    run_full_chain()
else:
    print('Using fast-demo mode (cached artifacts).')

Using fast-demo mode (cached artifacts).


In [ ]:
manifest = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'root': str(ROOT),
    'fast_demo_ready': bool(fast_demo_ready),
    'tier0_missing': [str(p) for p in m0],
    'tier1_missing': [str(p) for p in m1],
    'tier2_missing': [str(p) for p in m2],
    'notebook_readiness': notebook_status,
    'key_artifact_mtimes': {
        'signals_snapshot': (PROCESSED / 'signals_latest_snapshot.csv').stat().st_mtime if (PROCESSED / 'signals_latest_snapshot.csv').exists() else None,
        'ml_snapshot': (PROCESSED / 'ml_signals_snapshot.csv').stat().st_mtime if (PROCESSED / 'ml_signals_snapshot.csv').exists() else None,
        'ml_backtest': (PROCESSED / 'ml_backtest_summary.csv').stat().st_mtime if (PROCESSED / 'ml_backtest_summary.csv').exists() else None,
        'model_cmp_ru': (TABLES / 'tab_6_3_model_comparison_ru.csv').stat().st_mtime if (TABLES / 'tab_6_3_model_comparison_ru.csv').exists() else None,
        'advanced_methods': (TABLES / 'tab_6_4_advanced_methods.csv').stat().st_mtime if (TABLES / 'tab_6_4_advanced_methods.csv').exists() else None,
    }
}

PROCESSED.mkdir(parents=True, exist_ok=True)
manifest_path = PROCESSED / 'demo_run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Manifest saved: {manifest_path}')

Manifest saved: C:\Users\sotov\Documents\Interdependence of Ruble and Yuan Borrowing rates\data\processed\demo_run_manifest.json
